# Libraries

In [1]:
import random
import csv
import matplotlib.pyplot as plt
from collections import defaultdict

# Parameter Settings

In [2]:
GRID_SIZE = 50
TIME_STEPS = 200

MAX_PREY = 2000
MAX_PREDATORS = 500

INITIAL_PREY = 40
INITIAL_PREDATORS = 10

In [3]:
# AGENTS

class Prey:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class Predator:
    def __init__(self, x, y, energy=6):
        self.x = x
        self.y = y
        self.energy = energy

# WORLD

class World:
    def __init__(self, params):
        self.params = params
        self.prey = []
        self.predators = []

    def wrap(self, x, y):
        return x % GRID_SIZE, y % GRID_SIZE

    def build_grid(self):
        grid = defaultdict(list)
        for p in self.prey:
            grid[(p.x, p.y)].append(p)
        return grid

    # SENSING

    def sense_predators(self, prey):
        d = self.params["prey_sense"]
        return [p for p in self.predators
                if abs(p.x - prey.x) <= d and abs(p.y - prey.y) <= d]

    def sense_prey(self, predator):
        d = self.params["predator_sense"]
        return [pr for pr in self.prey
                if abs(pr.x - predator.x) <= d and abs(pr.y - predator.y) <= d]

    # MOVEMENT

    def move_prey(self):
        for prey in self.prey:
            predators = self.sense_predators(prey)

            moves = [(dx, dy)
                     for dx in range(-self.params["prey_move"], self.params["prey_move"] + 1)
                     for dy in range(-self.params["prey_move"], self.params["prey_move"] + 1)
                     if not (dx == 0 and dy == 0)]

            if predators:
                nearest = min(predators,
                              key=lambda p: abs(p.x - prey.x) + abs(p.y - prey.y))

                best_move = max(
                    moves,
                    key=lambda m: abs((prey.x + m[0]) - nearest.x) +
                                  abs((prey.y + m[1]) - nearest.y)
                )

                prey.x, prey.y = self.wrap(prey.x + best_move[0],
                                           prey.y + best_move[1])
            else:
                dx, dy = random.choice(moves)
                prey.x, prey.y = self.wrap(prey.x + dx, prey.y + dy)

    def move_predators(self):
        for predator in self.predators:
            prey_list = self.sense_prey(predator)

            moves = [(dx, dy)
                     for dx in range(-self.params["predator_move"], self.params["predator_move"] + 1)
                     for dy in range(-self.params["predator_move"], self.params["predator_move"] + 1)
                     if not (dx == 0 and dy == 0)]

            if prey_list:
                target = min(prey_list,
                             key=lambda pr: abs(pr.x - predator.x) +
                                            abs(pr.y - predator.y))

                best_move = min(
                    moves,
                    key=lambda m: abs((predator.x + m[0]) - target.x) +
                                  abs((predator.y + m[1]) - target.y)
                )

                predator.x, predator.y = self.wrap(predator.x + best_move[0],
                                                   predator.y + best_move[1])
            else:
                dx, dy = random.choice(moves)
                predator.x, predator.y = self.wrap(predator.x + dx,
                                                   predator.y + dy)

            predator.energy -= self.params["predator_move"]

    # INTERACTION (FAST)

    def eat(self):
        grid = self.build_grid()

        for predator in self.predators:
            cell = grid.get((predator.x, predator.y), [])
            if cell:
                victim = random.choice(cell)
                if victim in self.prey:
                    self.prey.remove(victim)
                    predator.energy += 5

    # REPRODUCTION

    def reproduce(self):
        new_prey = []

        for prey in self.prey:
            if len(self.prey) < MAX_PREY and random.random() < self.params["prey_reproduce"]:
                dx, dy = random.randint(-1, 1), random.randint(-1, 1)
                nx, ny = self.wrap(prey.x + dx, prey.y + dy)
                new_prey.append(Prey(nx, ny))

        self.prey.extend(new_prey)

        new_predators = []

        for predator in self.predators:
            if predator.energy >= self.params["predator_reproduce_energy"] \
               and len(self.predators) < MAX_PREDATORS:

                dx, dy = random.randint(-1, 1), random.randint(-1, 1)
                nx, ny = self.wrap(predator.x + dx, predator.y + dy)
                new_predators.append(Predator(nx, ny))
                predator.energy //= 2

        self.predators.extend(new_predators)

    # CLEANUP

    def remove_dead(self):
        self.predators = [p for p in self.predators if p.energy > 0]

# SIMULATION

def run_simulation(params, run_id):
    world = World(params)

    # INITIAL POPULATION
    for _ in range(INITIAL_PREY):
        world.prey.append(Prey(random.randint(0, 49), random.randint(0, 49)))

    for _ in range(INITIAL_PREDATORS):
        world.predators.append(Predator(random.randint(0, 49), random.randint(0, 49)))

    prey_counts = []
    predator_counts = []

    for step in range(TIME_STEPS):
        world.move_prey()
        world.move_predators()
        world.eat()
        world.remove_dead()
        world.reproduce()

        prey_counts.append(len(world.prey))
        predator_counts.append(len(world.predators))

    # SAVE CSV

    with open(f"task2_run_{run_id}.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Step", "Prey", "Predators"])
        for i in range(TIME_STEPS):
            writer.writerow([i, prey_counts[i], predator_counts[i]])

    # PLOT

    plt.figure()
    plt.plot(prey_counts, label="Prey")
    plt.plot(predator_counts, label="Predators")
    plt.xlabel("Time Step")
    plt.ylabel("Population")
    plt.title(f"Task 2 Run {run_id}")
    plt.legend()
    plt.savefig(f"task2_run_{run_id}.png")
    plt.close()



# Parameters

In [4]:
# PARAMETER SETTINGS

param_set_1 = {
    "prey_reproduce": 0.12,
    "predator_reproduce_energy": 12,
    "prey_sense": 3,
    "predator_sense": 3,
    "prey_move": 1,
    "predator_move": 2
}

param_set_2 = {
    "prey_reproduce": 0.10,
    "predator_reproduce_energy": 10,
    "prey_sense": 2,
    "predator_sense": 4,
    "prey_move": 1,
    "predator_move": 2
}

# Running Simulations

In [5]:
# RUN ALL 6 SIMULATIONS

run_id = 1

for params in [param_set_1, param_set_2]:
    for _ in range(3):
        print(f"Running simulation {run_id}...")
        run_simulation(params, run_id)
        run_id += 1

print("All simulations completed!")

Running simulation 1...
Running simulation 2...
Running simulation 3...
Running simulation 4...
Running simulation 5...
Running simulation 6...
All simulations completed!
